In [2]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.database import SalesDatabase

db = SalesDatabase()
db.connect()

Conectando ao banco: F:\Projetos Programação\Data Analysis\Sales Analysis\data\database\sales_analysis.db


In [3]:
#Visão geral dos pedidos por status

query = """
SELECT  order_status,
        COUNT(DISTINCT order_id) AS pedidos
FROM orders
GROUP BY order_status
ORDER BY pedidos DESC;
"""

db.execute_query(query)

,order_status,pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [4]:
#Receita e ticket médio por status do pedido

query = """
SELECT  o.order_status,
        COUNT(DISTINCT o.order_id) AS pedidos,
        ROUND(SUM(p.payment_value), 2) AS receita_total,
        ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS ticket_medio
FROM orders o
LEFT JOIN orders_payments p ON o.order_id = p.order_id
GROUP BY o.order_status
ORDER BY receita_total DESC;
"""

db.execute_query(query)

,order_status,pedidos,receita_total,ticket_medio
0,delivered,96478,15422461.77,159.85
1,shipped,1107,177213.96,160.08
2,canceled,625,143255.60,229.21
3,unavailable,609,126479.51,207.68
4,processing,301,69394.11,230.55
5,invoiced,314,69137.99,220.18
6,created,5,688.10,137.62
7,approved,2,241.08,120.54


In [5]:
#Receita por categoria 

query = """
WITH receita_por_categoria AS (
    SELECT  pr.product_category_name AS categoria,
            ROUND(SUM(i.total_value), 2) AS receita,
            COUNT(DISTINCT i.order_id) AS pedidos
    FROM order_items i
    LEFT JOIN products pr ON i.product_id = pr.product_id
    GROUP BY pr.product_category_name
)
SELECT  categoria,
        receita,
        ROUND(receita / pedidos, 2) AS ticket_medio_categoria
FROM receita_por_categoria
ORDER BY receita DESC
LIMIT 10;
"""

db.execute_query(query)

,categoria,receita,ticket_medio_categoria
0,beleza_saude,1441248.07,163.11
1,relogios_presentes,1305541.61,232.14
2,cama_mesa_banho,1241681.72,131.86
3,esporte_lazer,1156656.48,149.83
4,informatica_acessorios,1059272.40,158.36
5,moveis_decoracao,902511.79,139.95
6,utilidades_domesticas,778397.77,132.29
7,cool_stuff,719329.95,198.05
8,automotivo,685384.32,175.87
9,ferramentas_jardim,584219.21,166.07


In [6]:
#Evolução mensal com variação

query = """
WITH pedidos_mensais AS (
    SELECT  strftime('%Y-%m', order_purchase_timestamp) AS mes,
            COUNT(*) AS pedidos
    FROM orders
    GROUP BY strftime('%Y-%m', order_purchase_timestamp)
)
SELECT  mes,
        pedidos,
        LAG(pedidos) OVER (ORDER BY mes) AS pedidos_anteriores,
        pedidos - LAG(pedidos) OVER (ORDER BY mes) AS variacao,
        ROUND(100.0 * (pedidos - LAG(pedidos) OVER (ORDER BY mes)) /
        LAG(pedidos) OVER (ORDER BY mes), 2) AS variacao_pct
FROM pedidos_mensais
ORDER BY mes;
"""

db.execute_query(query)

,mes,pedidos,pedidos_anteriores,variacao,variacao_pct
0,2016-09,4,NaN,NaN,NaN
1,2016-10,324,4.0,320.0,8000.00
2,2016-12,1,324.0,-323.0,-99.69
3,2017-01,800,1.0,799.0,79900.00
4,2017-02,1780,800.0,980.0,122.50
5,2017-03,2682,1780.0,902.0,50.67
6,2017-04,2404,2682.0,-278.0,-10.37
7,2017-05,3700,2404.0,1296.0,53.91
8,2017-06,3245,3700.0,-455.0,-12.30
9,2017-07,4026,3245.0,781.0,24.07


In [7]:
#Rank de vendedores por receita
query = """
WITH receita_vendedores AS (
    SELECT  s.seller_id,
            s.seller_state,
            ROUND(SUM(i.total_value), 2) AS receita
    FROM order_items i
    LEFT JOIN sellers s ON i.seller_id = s.seller_id
    GROUP BY s.seller_id, s.seller_state
)
SELECT  seller_id,
        seller_state,
        receita,
        RANK() OVER (ORDER BY receita DESC) AS rank
FROM receita_vendedores
ORDER BY rank
LIMIT 10;   
"""

db.execute_query(query)

,seller_id,seller_state,receita,rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,249640.70,1
1,7c67e1448b00f6e969d365cea6b010ab,SP,239536.44,2
2,53243585a1d6dc2643021fd1853d8905,BA,235856.68,3
3,4a3ca9315b744ce9f8e9374361493884,SP,235539.96,4
4,fa1c13f2614d7b5c4749cbc52fecda94,SP,204084.73,5
5,da8622b14eb17ae2831f4ac5b9dab84a,SP,185192.32,6
6,7e93a43ef30c4f03f38b393420bc753a,SP,182754.05,7
7,1025f0e2d44d7041d6cf58b6550e0bfa,SP,172860.69,8
8,7a67c85e85bb2ce8582c35f2203ad736,SP,162648.38,9
9,955fee9216a65b617aa5c0531780ce60,SP,160602.68,10


In [8]:
#Rank vendedores por estado
query = """
WITH receita_vendedores AS (
    SELECT  s.seller_id,
            s.seller_state,
            ROUND(SUM(i.total_value), 2) AS receita
    FROM order_items i
    LEFT JOIN sellers s ON i.seller_id = s.seller_id
    GROUP BY s.seller_id, s.seller_state
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY seller_state ORDER BY receita DESC) AS pos
    FROM receita_vendedores
)
SELECT seller_id, seller_state, receita, pos
FROM ranked
WHERE pos = 1
ORDER BY receita DESC;
"""

db.execute_query(query)

,seller_id,seller_state,receita,pos
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,249640.70,1
1,53243585a1d6dc2643021fd1853d8905,BA,235856.68,1
2,46dc3b2cc0980fb8ec44634e21d2718e,RJ,139909.69,1
3,a1043bafd471dff536d0c462352beb48,MG,133745.25,1
4,ccc4bbb5f32a6ab2b7066a4130f114e3,PR,79261.60,1
5,de722cd6dad950a92b7d4f82673f8833,PE,65112.75,1
6,04308b1ee57b6625f47df1d56f00eedf,SC,63184.99,1
7,06a2c3af7b3aee5d69171b0e14f0ee87,MA,48550.24,1
8,87142160b41353c4e5fca2360caf6f92,RS,38314.31,1
9,001cca7ae9ae17fb1caed9dfb1094831,ES,33934.17,1


In [12]:
#Percentual de entrega no prazo

query = """
WITH entregas AS (
    SELECT  o.order_id,
            CASE WHEN date(o.order_delivered_customer_date) <= date(o.order_estimated_delivery_date) 
            THEN 1 ELSE 0
            END AS no_prazo
    FROM orders o
    WHERE o.order_delivered_customer_date IS NOT NULL
    )
SELECT  pr.product_category_name AS categoria,
        COUNT(DISTINCT e.order_id) AS pedidos,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN e.no_prazo = 1 THEN e.order_id END) /
            COUNT(DISTINCT e.order_id), 2) AS pct_no_prazo
FROM entregas e
JOIN order_items i ON e.order_id = i.order_id
LEFT JOIN products pr ON i.product_id = pr.product_id
WHERE pr.product_category_name IS NOT NULL
GROUP BY pr.product_category_name
HAVING pedidos >= 50
ORDER BY pct_no_prazo DESC
LIMIT 10;
"""

db.execute_query(query)

,categoria,pedidos,pct_no_prazo
0,construcao_ferramentas_seguranca,167,97.60
1,alimentos_bebidas,227,96.48
2,livros_importados,53,96.23
3,agro_industria_e_comercio,182,96.15
4,market_place,280,96.07
5,climatizacao,253,96.05
6,malas_acessorios,1034,95.74
7,sinalizacao_e_seguranca,140,95.71
8,eletrodomesticos,764,95.68
9,moveis_cozinha_area_de_servico_jantar_e_jardim,248,95.56


In [10]:
#Análise de Notas por Estado do CLiente 

query = """
SELECT  c.customer_state AS estado,
        ROUND(AVG(r.review_score), 2) AS nota_media,
        COUNT(DISTINCT r.review_id) AS avaliacoes
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_reviews r ON o.order_id = r.order_id
GROUP BY c.customer_state
HAVING avaliacoes >= 30
ORDER BY nota_media DESC;
"""

db.execute_query(query)

,estado,nota_media,avaliacoes
0,AP,4.19,67
1,SP,4.18,41360
2,PR,4.18,5004
3,AM,4.18,146
4,RS,4.14,5430
5,MG,4.14,11526
6,MS,4.11,713
7,TO,4.10,277
8,RN,4.10,479
9,MT,4.10,898
